In [1]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git clone https://{GITHUB_TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
data_dir = "data"
while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

%pip install -q dagshub
!pip install mlflow xgboost --quiet

import dagshub
import mlflow

dagshub.init(repo_owner="amama22", repo_name="walmart-forecasting", mlflow=True)

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 98 (delta 47), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 322.86 KiB | 5.98 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/walmart-forecasting
100% 2.70M/2.70M [00:00<00:00, 215MB/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=72b2777a-0e07-4d87-a551-89fdc13acd37&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b9206815d3cef1548782f062c9d97dceefa615334601129471a391809f5dedfa




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

In [2]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

from src.data_prep import load_raw_data, merge_all, clean_data
from src.feature_engineering import add_holiday_proximity

train, test, features, stores = load_raw_data(data_dir="data")

PRUNED_FEATURE_COLS = [
    "Store", "Dept", "Type", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown3",
    "Month", "WeekOfYear",
    "sales_lag_52wk",
    "IsThanksgiving",
    "days_to_nearest_holiday",
    "dept_avg_sales_expanding", "store_avg_sales_expanding",
]

class WalmartFeatureBuilder(BaseEstimator, TransformerMixin):
    """Must match the XGBoost training notebook's class exactly -- same name,
    same methods -- so cloudpickle can resolve it correctly on load."""

    def __init__(self, features_df, stores_df):
        self.features_df = features_df
        self.stores_df = stores_df

    def fit(self, X, y=None):
        df = X.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df["Year"] = df["Date"].dt.year
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

        if y is not None:
            df["Weekly_Sales"] = np.asarray(y)

        self.history_ = df[["Store", "Dept", "Year", "WeekOfYear", "Weekly_Sales"]].copy()

        dept_weekly = df.groupby(["Dept", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Dept", "Date"])
        self.dept_avg_final_ = (
            dept_weekly.groupby("Dept")
            .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean(), include_groups=False)
        ).to_dict()

        store_weekly = df.groupby(["Store", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Store", "Date"])
        self.store_avg_final_ = (
            store_weekly.groupby("Store")
            .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean(), include_groups=False)
        ).to_dict()

        return self

    def transform(self, X):
        df = merge_all(X, self.features_df, self.stores_df)
        df = clean_data(df)

        df["Date"] = pd.to_datetime(df["Date"])
        df["Month"] = df["Date"].dt.month
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
        df["Year"] = df["Date"].dt.year

        thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
        df["IsThanksgiving"] = df["Date"].isin(thanksgiving)

        lookup = self.history_.copy()
        lookup["Year"] = lookup["Year"] + 1
        lookup = lookup.rename(columns={"Weekly_Sales": "sales_lag_52wk"})
        df = df.merge(lookup, on=["Store", "Dept", "Year", "WeekOfYear"], how="left")

        df = add_holiday_proximity(df)

        df["dept_avg_sales_expanding"] = df["Dept"].map(self.dept_avg_final_)
        df["store_avg_sales_expanding"] = df["Store"].map(self.store_avg_final_)

        for col in ["Store", "Dept", "Type"]:
            df[col] = df[col].astype("category")

        return df[PRUNED_FEATURE_COLS]

print("Test set shape:", test.shape)
test.head()

Test set shape: (115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [3]:
model_uri = "models:/walmart-xgboost-store-sales/1"
loaded_pipeline = mlflow.sklearn.load_model(model_uri)

print("Loaded pipeline steps:", loaded_pipeline.named_steps)

Loaded pipeline steps: {'features': WalmartFeatureBuilder(features_df=      Store       Date  Temperature  Fuel_Price  MarkDown1  MarkDown2  \
0         1 2010-02-05        42.31       2.572        NaN        NaN   
1         1 2010-02-12        38.51       2.548        NaN        NaN   
2         1 2010-02-19        39.93       2.514        NaN        NaN   
3         1 2010-02-26        46.63       2.561        NaN        NaN   
4         1 2010-03-05        46.50       2.625        NaN        NaN   
...     ...        ...          ...         ...        ...        ...   
8185     45 2013-06-28        76.05       3.639    4842.29     975.03   
8186     45 2013-07-05        77.50       3.614    9090.48    2268.58   
8187     45 2013-07-12        79.37       3.614    3...
12     13    A  219622
13     14    A  200898
14     15    B  123737
15     16    B   57197
16     17    B   93188
17     18    B  120653
18     19    A  203819
19     20    A  203742
20     21    B  140167
21     22 

In [4]:
test_preds = loaded_pipeline.predict(test)
test_preds = np.clip(test_preds, 0, None)  # Weekly_Sales can't be negative

submission = pd.DataFrame({
    "Id": test["Store"].astype(str) + "_" + test["Dept"].astype(str) + "_" + test["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_preds,
})

print("Submission shape:", submission.shape, "| expected:", test.shape[0])
submission.to_csv("submission_final.csv", index=False)
submission.head()

Submission shape: (115064, 2) | expected: 115064


,Id,Weekly_Sales
0,1_1_2012-11-02,40448.820312
1,1_1_2012-11-09,20543.042969
2,1_1_2012-11-16,21516.287109
3,1_1_2012-11-23,21224.658203
4,1_1_2012-11-30,26529.964844


In [5]:
mlflow.set_experiment("Model_Inference")

with mlflow.start_run(run_name="XGBoost_Final_Inference"):
    mlflow.log_param("model_uri", model_uri)
    mlflow.log_param("n_predictions", len(submission))
    mlflow.log_metric("mean_predicted_sales", float(submission["Weekly_Sales"].mean()))
    mlflow.log_artifact("submission_final.csv")

print("Logged inference run.")

2026/07/12 17:34:05 INFO mlflow.tracking.fluent: Experiment with name 'Model_Inference' does not exist. Creating a new experiment.


🏃 View run XGBoost_Final_Inference at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/5/runs/95e573b3fd914d08ab7c6c21912f6188
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/5
Logged inference run.


In [6]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_final.csv -m "Final inference notebook, XGBoost from Model Registry, registry round-trip check"

100% 2.87M/2.87M [00:00<00:00, 3.95MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting